# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

---


Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками


In [3]:
#!wget https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4 -O data/slow_traffic_small.mp4

In [4]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:**

- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV


### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**

1. Яркость сохраняется

Для одной и той же точки сцены интенсивность почти не меняется между кадрами. Важно так как это дает основное уравнение оптического потока

2. Малое смещение между соседними кадрами

Сдвиг должен быть небольшим чтобы линейное разложение Тейлора было корректным. Иначе аппроксимация I(x+u,y+v) через градиенты становится неточной.

3. Локально постоянный поток в окне

В небольшой окрестности точки все пиксели движутся примерно одинаково. Это позволяет устойчиво оценить движение.


### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:**

Пирамидальный подход нужен, чтобы обрабатывать большие смещения. Сначала оценка на грубом (уменьшенном) уровне, где тот же физический сдвиг в пикселях становится меньше и малое смещение снова работает. Затем оценка последовательно уточняется на более детальных уровнях.


### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**

1. На однородных областях или одномерных текстурах движение неоднозначно.

2. Большие межкадровые смещения без пирамиды приводят к срыву трекинга.

3. Изменение яркости/освещения нарушает предположение сохранения интенсивности.

4. Шум, blur, компрессия ухудшают градиенты и стабильность решения.


### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.


In [5]:
def build_image_pyramid(image, num_levels, scale_factor=0.5):
    """
    Создаёт пирамиду изображений с уменьшающимся разрешением.

    Аргументы:
        image: Исходное изображение (одноканальное, grayscale)
        num_levels: Количество уровней пирамиды
        scale_factor: Коэффициент масштабирования между соседними уровнями

    Возвращает:
        Список изображений [image_level_0, image_level_1, ..., image_level_n-1], где:
        - image_level_0 - исходное изображение
        - Каждый следующий уровень уменьшен относительно предыдущего в scale_factor раз

    Примечание:
        - Используйте cv2.resize с интерполяцией cv2.INTER_LINEAR для уменьшения размера
        - Первым элементом списка должна быть копия исходного изображения
    """
    pyramid = [image.copy()]
    current = image.copy()
    for _ in range(1, num_levels):
        h, w = current.shape[:2]
        new_w = max(1, int(w * scale_factor))
        new_h = max(1, int(h * scale_factor))
        current = cv2.resize(current, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        pyramid.append(current)
    return pyramid


def compute_image_gradients(image):
    """
    Вычисляет пространственные градиенты изображения.

    Аргументы:
        image: Входное изображение (одноканальное, grayscale)

    Возвращает:
        Кортеж (Ix, Iy), где Ix и Iy - градиенты по x и y соответственно

    Примечание:
        - Используйте фильтр Собеля (cv2.Sobel) с ksize=3
        - Используйте тип данных cv2.CV_64F для более точных вычислений
    """
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix, Iy, It, window_size=5):
    """
    Вычисляет оптический поток по методу Lucas-Kanade для одного окна.

    Аргументы:
        Ix: Градиент изображения по x
        Iy: Градиент изображения по y
        It: Временной градиент (разница между кадрами)
        window_size: Размер окна для вычисления (нечетное число)

    Возвращает:
        Кортеж (u, v) компонентов вектора потока или (None, None) если решение ненадежное

    Примечание:
        - Создайте окно для градиентов, выбрав центральный пиксель и окно размером window_size x window_size
        - Вычислите сумму произведений градиентов для формирования матрицы A:
          A = [[sum(Ix*Ix), sum(Ix*Iy)], [sum(Ix*Iy), sum(Iy*Iy)]]
        - Проверьте обусловленность матрицы A через собственные значения
        - Если минимальное собственное значение меньше порога (например, 1e-4), верните (None, None)
        - Сформируйте вектор b: [-sum(Ix*It), -sum(Iy*It)]
        - Решите систему уравнений A * [u, v] = b
        - Обработайте возможное исключение np.linalg.LinAlgError
    """
    h, w = Ix.shape
    
    half = window_size // 2
    cy, cx = h // 2, w // 2
    
    if cy - half < 0 or cy + half >= h or cx - half < 0 or cx + half >= w:
        return None, None
    
    ix = Ix[cy - half:cy + half + 1, cx - half:cx + half + 1].reshape(-1)
    iy = Iy[cy - half:cy + half + 1, cx - half:cx + half + 1].reshape(-1)
    it = It[cy - half:cy + half + 1, cx - half:cx + half + 1].reshape(-1)
    
    A = np.array([
        [np.sum(ix * ix), np.sum(ix * iy)],
        [np.sum(ix * iy), np.sum(iy * iy)]
    ], dtype=np.float64)

    eigvals = np.linalg.eigvalsh(A)
    
    if np.min(eigvals) < 1e-4:
        return None, None
    
    b = np.array([
        -np.sum(ix * it),
        -np.sum(iy * it)
    ], dtype=np.float64)
    
    try:
        u, v = np.linalg.solve(A, b)
        return float(u), float(v)
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    """
    Вычисляет оптический поток для патча изображения.

    Аргументы:
        prev_patch: Патч из предыдущего кадра
        curr_patch: Соответствующий патч из текущего кадра
        window_size: Размер окна для LK

    Возвращает:
        Кортеж (u, v) компонентов вектора потока для центра патча

    Примечание:
        - Вычислите пространственные градиенты prev_patch с помощью compute_image_gradients
        - Вычислите временной градиент как разность патчей: It = curr_patch - prev_patch
        - Используйте функцию compute_lk_optical_flow_point для вычисления вектора потока
    """
    Ix, Iy = compute_image_gradients(prev_patch)
    It = curr_patch.astype(np.float64) - prev_patch.astype(np.float64)
    return compute_lk_optical_flow_point(Ix, Iy, It, window_size=window_size)


def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    """
    Отслеживает точку между кадрами с использованием пирамидального LK.

    Аргументы:
        prev_pyramid: Пирамида предыдущего кадра (список изображений)
        curr_pyramid: Пирамида текущего кадра (список изображений)
        point: Координаты отслеживаемой точки (x, y) на исходном изображении
        window_size: Размер окна для LK
        max_iterations: Максимальное количество итераций для уточнения каждого уровня
        epsilon: Порог для остановки итераций

    Возвращает:
        Кортеж (new_x, new_y) - новые координаты точки на текущем кадре
        или None если отслеживание неуспешно

    Примечание:
        - Начните обработку с верхнего уровня пирамиды (самое маленькое изображение)
        - Масштабируйте исходную точку для соответствия размеру изображения верхнего уровня
        - Для каждого уровня пирамиды (от верхнего к нижнему):
            1. Масштабируйте общее смещение в 2 раза при переходе на уровень ниже
            2. Пересчитайте позицию точки с учетом масштаба текущего уровня
            3. Примените итеративное уточнение позиции с помощью LK:
                a. Проверьте, что точка и окно вокруг неё находятся в границах изображения
                b. Извлеките патчи из предыдущего и текущего кадров
                c. Вычислите смещение с помощью compute_lk_optical_flow_for_patch
                d. Обновите позицию точки
                e. Остановите итерации, если смещение меньше epsilon
            4. Обновите общее смещение
        - Вычислите финальную позицию точки на исходном изображении
    """
    levels = len(prev_pyramid)
    x0, y0 = float(point[0]), float(point[1])
    
    # стартуем с самого грубого уровня
    scale_top = 0.5 ** (levels - 1)
    
    x = x0 * scale_top
    y = y0 * scale_top
    
    for lvl in range(levels - 1, -1, -1):
        prev_img = prev_pyramid[lvl]
        curr_img = curr_pyramid[lvl]
        h, w = prev_img.shape[:2]
        half = window_size // 2
        
        if lvl < levels - 1:
            x *= 2.0
            y *= 2.0
        
        for _ in range(max_iterations):
            xi, yi = int(round(x)), int(round(y))
            
            if (xi - half < 0 or xi + half >= w or yi - half < 0 or yi + half >= h):
                return None
            
            prev_patch = prev_img[yi - half:yi + half + 1, xi - half:xi + half + 1]
            
            # текущий патч берем в той же позиции (итеративное уточнение локального смещения)
            curr_patch = curr_img[yi - half:yi + half + 1, xi - half:xi + half + 1]
            du, dv = compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=window_size)
            
            if du is None or dv is None:
                return None
            x_new = x + du
            y_new = y + dv
            
            if np.hypot(du, dv) < epsilon:
                x, y = x_new, y_new
                break
            x, y = x_new, y_new
        
        # проверка после итераций
        if not (0 <= x < w and 0 <= y < h):
            return None
    
    return float(x), float(y)


def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    """
    Вычисляет разреженный оптический поток методом Лукаса-Канаде.

    Аргументы:
        prev_frame: Предыдущий кадр (может быть цветным)
        curr_frame: Текущий кадр (может быть цветным)
        points: Список точек для отслеживания в формате [[x1, y1], [x2, y2], ...]
        window_size: Размер окна для LK
        num_pyramid_levels: Количество уровней в пирамиде изображений
        max_iterations: Максимальное количество итераций на каждом уровне
        epsilon: Порог сходимости для итераций

    Возвращает:
        Кортеж (new_points, status), где:
        - new_points: Массив новых позиций точек в формате [[x1, y1], [x2, y2], ...]
        - status: Массив статусов отслеживания (1 - успешно, 0 - неуспешно)

    Примечание:
        - Преобразуйте входные кадры в полутоновые, если они цветные
        - Нормализуйте изображения к диапазону [0, 1]
        - Создайте пирамиды изображений для обоих кадров
        - Для каждой точки из списка:
            1. Отследите её с помощью track_point_with_pyramid_lk
            2. Сохраните результат в new_points и отметьте статус в status
        - Если отслеживание точки не удалось, установите статус 0 и сохраните исходную точку
    """
    if prev_frame.ndim == 3:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    else:
        prev_gray = prev_frame.copy()
    
    if curr_frame.ndim == 3:
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    else:
        curr_gray = curr_frame.copy()
    
    prev_gray = prev_gray.astype(np.float32) / 255.0
    curr_gray = curr_gray.astype(np.float32) / 255.0
    
    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels, scale_factor=0.5)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels, scale_factor=0.5)
    
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    
    new_points = points.copy()
    status = np.zeros((len(points),), dtype=np.uint8)
    
    for i, pt in enumerate(points):
        tracked = track_point_with_pyramid_lk(
            prev_pyramid, curr_pyramid, pt,
            window_size=window_size,
            max_iterations=max_iterations,
            epsilon=epsilon
        )
        
        if tracked is not None:
            new_points[i] = tracked
            status[i] = 1
        else:
            new_points[i] = pt
            status[i] = 0
    
    return new_points, status


def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)  # Преобразуем в формат [[x1, y1], [x2, y2], ...]

    # Сохраняем изначальные точки для отслеживания через все видео
    initial_points = p0.copy()

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью нашей реализации
        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3
        )

        # Выбираем хорошие точки
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0[st == 1] = good_new

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [6]:
result_path = demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:30<00:00, 29.68it/s]

Результат сохранен в output_my_LK.mp4


### Релизация OpenCV - cv2.calcOpticalFlowPyrLK


In [7]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео с использованием cv2.calcOpticalFlowPyrLK.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Параметры для Lucas-Kanade оптического потока
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью встроенной функции cv2.calcOpticalFlowPyrLK
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Выбираем хорошие точки
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0 = good_new.reshape(-1, 1, 2)

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [8]:
result_path = demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:03<00:00, 235.97it/s]

Результат сохранен в output_opencv_LK.mp4


### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.


In [9]:
# your code

def detect_new_points(frame_gray, existing_points, max_corners=120, quality_level=0.3, min_distance=7, block_size=7):
    """
    Ищет новые Shi-Tomasi точки, избегая областей рядом с уже отслеживаемыми точками.
    """
    h, w = frame_gray.shape[:2]
    mask = np.full((h, w), 255, dtype=np.uint8)

    # Запрещаем искать новые точки рядом с уже существующими
    if existing_points is not None and len(existing_points) > 0:
        for x, y in existing_points:
            cv2.circle(mask, (int(x), int(y)), int(min_distance * 1.5), 0, -1)

    new_pts = cv2.goodFeaturesToTrack(
        frame_gray,
        mask=mask,
        maxCorners=max_corners,
        qualityLevel=quality_level,
        minDistance=min_distance,
        blockSize=block_size
    )

    if new_pts is None:
        return np.empty((0, 2), dtype=np.float32)

    return new_pts.reshape(-1, 2).astype(np.float32)


def merge_points(old_points, tracked_points, status, frame_gray, target_points=120):
    """
    Удаляет потерянные точки и добавляет новые до целевого количества.
    """
    if tracked_points is None or len(tracked_points) == 0:
        alive = np.empty((0, 2), dtype=np.float32)
    else:
        alive = tracked_points[status == 1].astype(np.float32)

    need = max(0, target_points - len(alive))
    if need > 0:
        new_pts = detect_new_points(
            frame_gray,
            alive,
            max_corners=need,
            quality_level=0.3,
            min_distance=7,
            block_size=7
        )
        if len(new_pts) > 0:
            alive = np.vstack([alive, new_pts]) if len(alive) > 0 else new_pts

    return alive.astype(np.float32)


def demo_optical_flow_with_point_recovery(video_path='data/slow_traffic_small.mp4',
                                          output_path='output_my_LK_recovery.mp4',
                                          target_points=120):
    """
    Lucas-Kanade + механизм восстановления точек:
    - удаляем потерянные;
    - дозаполняем новыми точками на текущем кадре.
    """
    cap = cv2.VideoCapture(video_path)

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    ret, old_frame = cap.read()
    if not ret:
        cap.release()
        out.release()
        raise RuntimeError("Не удалось прочитать первый кадр")

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = detect_new_points(old_gray, existing_points=None, max_corners=target_points)

    mask_tracks = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (2000, 3), dtype=np.uint8)

    from tqdm import tqdm
    for _ in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Если точек нет — просто переинициализируем
        if p0 is None or len(p0) == 0:
            p0 = detect_new_points(frame_gray, existing_points=None, max_corners=target_points)
            old_gray = frame_gray.copy()
            out.write(frame)
            continue

        # Трекинг существующих точек
        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3,
            max_iterations=10,
            epsilon=0.01
        )

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # Визуализация треков только для выживших точек
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            col = color[i % len(color)].tolist()
            mask_tracks = cv2.line(mask_tracks, (int(c), int(d)), (int(a), int(b)), col, 2)
            frame = cv2.circle(frame, (int(a), int(b)), 4, col, -1)

        # Обновляем набор точек: удаляем потерянные + добавляем новые
        p0 = merge_points(
            old_points=p0,
            tracked_points=p1,
            status=st,
            frame_gray=frame_gray,
            target_points=target_points
        )

        img = cv2.add(frame, mask_tracks)
        out.write(img)
        old_gray = frame_gray.copy()

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [10]:
result_path = demo_optical_flow_with_point_recovery(
    video_path='data/slow_traffic_small.mp4',
    output_path='output_my_LK_recovery.mp4',
    target_points=120
)

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [04:35<00:00,  3.31it/s]

Результат сохранен в output_my_LK_recovery.mp4


### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:**

1. Sparse LK: считает поток только для выбранных ключевых точек (углы/фичи), быстрее и дешевле, удобен для трекинга признаков.

2. Dense (Farneback): оценивает вектор движения для каждого пикселя, дает полное поле потока, но обычно тяжелее по вычислениям и памяти.


## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.


# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**

1. Построение гауссовой пирамиды кадров

2. В каждой локальной области аппроксимация сигнала квадратичным полиномом

3. Сопоставление полиномиальных моделей двух кадров для оценки локального смещения

4. Итеративное уточнение потока на каждом уровне

5. Переход от грубого уровня к более детальному с распространением оценки

6. Получение плотного поля (u,v) для всех пикселей


### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:**

Большие смещения в Farneback обрабатываются за счет пирамидального подхода:

- на верхних (уменьшенных) уровнях большие реальные движения становятся малыми в пикселях;

- сначала оценивается грубый поток;

- затем он масштабируется вниз и уточняется на более детальных уровнях


In [ ]:
def demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_Farneback.mp4'):
    """
    Демонстрация работы алгоритма плотного оптического потока Farneback на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Берем первый кадр и преобразуем его в оттенки серого
    ret, frame1 = cap.read()
    if not ret:
        print('Не удалось прочитать видео')
        return None

    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    # Создаем HSV-изображение для визуализации потока
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255  # Насыщенность устанавливаем на максимум

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame2 = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток методом Farneback
        # Параметры:
        # - 0.5: коэффициент масштабирования для пирамиды изображений
        # - 3: кол-во уровней пирамиды
        # - 15: размер окна для усреднения
        # - 3: число итераций на каждом уровне пирамиды
        # - 5: размер окна для полиномиальной аппроксимации
        # - 1.2: стандартное отклонение для сглаживания
        flow = cv2.calcOpticalFlowFarneback(
            prvs, next_frame, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Преобразуем векторы потока из декартовых координат в полярные
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Кодируем направление потока как оттенок (hue)
        hsv[..., 0] = ang * 180 / np.pi / 2

        # Кодируем величину потока как яркость (value)
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        # Преобразуем HSV в BGR для отображения
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Записываем результат
        out.write(bgr)

        # Обновляем предыдущий кадр
        prvs = next_frame

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_farneback.mp4')

### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:**

Предобработка сильно влияет на качество потока:

- подавление шума стабилизирует градиенты и уменьшает ложные векторы

- выравнивание контраста повышает информативность в слаботекстурных участках

- но агрессивная обработка может исказить движение

Оптимальный пайплайн предобработки:

- Перевод в grayscale

- Легкое шумоподавление

- Нормализация освещения

- нормализация интенсивности при сильной засветке

- подача в calcOpticalFlowFarneback с подбором параметров под сцену

- медианный фильтр по полю потока для удаления выбросов
